# 04b - Recomendador con generos y tags

Este notebook crea una mejora opcional del recomendador basado en contenido combinando los generos de las peliculas con las etiquetas (`tags`) introducidas por los usuarios. La idea es obtener una representacion mas rica del contenido sin complicar demasiado el modelo.

## 1. Por que pueden aportar valor los tags

Los generos ofrecen una descripcion general de la pelicula, pero a veces son demasiado amplios. Los tags pueden a?adir matices como temas, tono, estilo o elementos concretos que no aparecen en los generos.

Por ejemplo, dos peliculas pueden compartir el genero `Sci-Fi`, pero una puede estar etiquetada con ideas como `time travel` y otra con `cyberpunk`. Por eso los tags pueden enriquecer el recomendador.

Aun asi, tambien tienen limitaciones: no todas las peliculas tienen tags y la calidad de las etiquetas depende de lo que hayan escrito los usuarios.

## 2. Importacion de librerias

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

base_dir = Path('..').resolve()
sys.path.append(str(base_dir))

from src.data_utils import load_tags
from src.recommender import (
    build_genre_feature_matrix,
    find_movie_index,
    get_genre_columns,
    recommend_movies_by_genres,
)

## 3. Carga de datos

In [2]:
movies_path = base_dir / 'data' / 'processed' / 'movies_clean.csv'
tags_path = base_dir / 'data' / 'raw' / 'tags.csv'

movies_clean = pd.read_csv(movies_path)
tags_df = load_tags(tags_path)

print('movies_clean shape:', movies_clean.shape)
print('tags_df shape:', tags_df.shape)

movies_clean shape: (87585, 26)
tags_df shape: (2000072, 4)


## 4. Limpieza de tags

Primero limpiamos las etiquetas: pasamos el texto a minusculas, eliminamos nulos y agrupamos todas las etiquetas de una misma pelicula.

In [3]:
tags_clean = tags_df[['movieId', 'tag']].copy()
tags_clean['tag'] = tags_clean['tag'].fillna('').str.lower().str.strip()
tags_clean = tags_clean[tags_clean['tag'] != '']

tags_grouped = (
    tags_clean.groupby('movieId')['tag']
    .apply(lambda values: ' '.join(values))
    .reset_index(name='tags_combined')
)

display(tags_grouped.head())

,movieId,tags_combined
0,1,children disney animation children disney disn...
1,2,robin williams fantasy robin williams time tra...
2,3,comedinha de velhinhos engraãƒâ§ada comedinha ...
3,4,characters slurs based on novel or book chick ...
4,5,fantasy pregnancy remake family steve martin s...


## 5. Union con la tabla de peliculas

Ahora unimos la informacion de tags con `movies_clean` para que cada pelicula tenga, cuando sea posible, una cadena con todas sus etiquetas.

In [4]:
movies_with_tags = movies_clean.merge(tags_grouped, on='movieId', how='left')
movies_with_tags['tags_combined'] = movies_with_tags['tags_combined'].fillna('')

tagged_movies = (movies_with_tags['tags_combined'] != '').sum()
tag_coverage = tagged_movies / len(movies_with_tags)

print('Peliculas con tags:', tagged_movies)
print('Cobertura de tags:', round(tag_coverage, 3))

Peliculas con tags: 51323
Cobertura de tags: 0.586


## 6. Creacion de content_features

Combinamos generos y tags en una sola columna de texto. De este modo, la pelicula queda representada tanto por categorias generales como por detalles adicionales.

In [5]:
movies_with_tags['genres_text'] = movies_with_tags['genres'].fillna('').str.replace('|', ' ', regex=False)
movies_with_tags['content_features'] = (
    movies_with_tags['genres_text'].str.strip() + ' ' + movies_with_tags['tags_combined'].str.strip()
).str.strip()

display(movies_with_tags[['title', 'genres', 'tags_combined', 'content_features']].head())

,title,genres,tags_combined,content_features
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,children disney animation children disney disn...,Adventure Animation Children Comedy Fantasy ch...
1,Jumanji (1995),Adventure|Children|Fantasy,robin williams fantasy robin williams time tra...,Adventure Children Fantasy robin williams fant...
2,Grumpier Old Men (1995),Comedy|Romance,comedinha de velhinhos engraãƒâ§ada comedinha ...,Comedy Romance comedinha de velhinhos engraãƒâ...
3,Waiting to Exhale (1995),Comedy|Drama|Romance,characters slurs based on novel or book chick ...,Comedy Drama Romance characters slurs based on...
4,Father of the Bride Part II (1995),Comedy,fantasy pregnancy remake family steve martin s...,Comedy fantasy pregnancy remake family steve m...


## 7. Vectorizacion del contenido

Para convertir el texto en una representacion numerica usamos `TfidfVectorizer`. Asi damos mas peso a palabras informativas y menos a terminos demasiado frecuentes.

In [6]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english', min_df=2)
content_matrix = tfidf_vectorizer.fit_transform(movies_with_tags['content_features'])

print('Forma de la matriz TF-IDF:', content_matrix.shape)

Forma de la matriz TF-IDF: (87585, 28305)


## 8. Similitud coseno

La recomendacion tambien se basara en similitud coseno, pero ahora sobre una representacion mas rica. Como el dataset es grande, calcularemos la similitud de una pelicula frente al resto cuando la necesitemos, en lugar de guardar una matriz completa en memoria.

In [7]:
example_similarity = cosine_similarity(content_matrix[0], content_matrix).flatten()
print('Numero de similitudes calculadas para la pelicula de ejemplo:', len(example_similarity))
print('Primeras similitudes:', example_similarity[:5])

Numero de similitudes calculadas para la pelicula de ejemplo: 87585
Primeras similitudes: [1.         0.07117875 0.03507157 0.0051532  0.04148603]


## 9. Funcion de recomendacion con generos y tags

La funcion siguiente usa la nueva representacion combinada y mantiene el filtro opcional por numero minimo de valoraciones para que los resultados sigan siendo faciles de interpretar.

In [8]:
def recommend_movies_by_content(movie_title, n=10, min_ratings=20):
    movie_index = find_movie_index(movies_with_tags, movie_title)
    if movie_index is None:
        return pd.DataFrame(columns=[
            'title',
            'year',
            'genres',
            'rating_mean',
            'rating_count',
            'similarity_score',
        ])

    similarity_scores = cosine_similarity(content_matrix[movie_index], content_matrix).flatten()
    candidate_indices = similarity_scores.argsort()[::-1]
    candidate_indices = [idx for idx in candidate_indices if idx != movie_index]

    recommendations = movies_with_tags.loc[candidate_indices, [
        'title_clean', 'title', 'year', 'genres', 'rating_mean', 'rating_count'
    ]].copy()
    recommendations['title'] = recommendations['title_clean'].fillna(recommendations['title'])
    recommendations['similarity_score'] = similarity_scores[candidate_indices]
    recommendations = recommendations[[
        'title', 'year', 'genres', 'rating_mean', 'rating_count', 'similarity_score'
    ]]
    recommendations = recommendations.sort_values(
        ['similarity_score', 'rating_count', 'rating_mean'],
        ascending=[False, False, False],
    )

    if min_ratings > 0:
        high_confidence = recommendations[recommendations['rating_count'] >= min_ratings]
        fallback = recommendations[recommendations['rating_count'] < min_ratings]
        recommendations = pd.concat([high_confidence, fallback], ignore_index=True)
        if len(high_confidence) < n:
            print('No hay suficientes peliculas con el minimo de valoraciones indicado. Se completan los resultados con peliculas menos valoradas.')

    print(f"Pelicula seleccionada: {movies_with_tags.loc[movie_index, 'title']}")
    return recommendations.head(n).reset_index(drop=True)

## 10. Recomendador solo con generos

Para comparar ambos enfoques, primero obtenemos recomendaciones con la version basada solo en generos.

In [9]:
genre_columns = get_genre_columns(movies_clean)
genre_feature_matrix = build_genre_feature_matrix(movies_clean, genre_columns)

genre_recommendations = recommend_movies_by_genres(
    movies_clean,
    genre_feature_matrix,
    'Toy Story',
    n=5,
    min_ratings=20,
)
genre_recommendations

Pelicula seleccionada: Toy Story (1995)


,title,year,genres,rating_mean,rating_count,similarity_score
0,"Monsters, Inc.",2001.0,Adventure|Animation|Children|Comedy|Fantasy,3.837442,46036,1.0
1,Toy Story 2,1999.0,Adventure|Animation|Children|Comedy|Fantasy,3.812043,32683,1.0
2,Antz,1998.0,Adventure|Animation|Children|Comedy|Fantasy,3.213386,12857,1.0
3,"Emperor's New Groove, The",2000.0,Adventure|Animation|Children|Comedy|Fantasy,3.650783,11437,1.0
4,Moana,2016.0,Adventure|Animation|Children|Comedy|Fantasy,3.816381,8278,1.0


## 11. Recomendador con generos y tags

Ahora aplicamos la nueva version para el mismo titulo y observamos si aparecen peliculas mas especificas o con matices distintos.

In [10]:
content_recommendations = recommend_movies_by_content('Toy Story', n=5, min_ratings=20)
content_recommendations

Pelicula seleccionada: Toy Story (1995)


,title,year,genres,rating_mean,rating_count,similarity_score
0,Toy Story 2,1999.0,Adventure|Animation|Children|Comedy|Fantasy,3.812043,32683,0.888007
1,"Bug's Life, A",1998.0,Adventure|Animation|Children|Comedy,3.558105,26736,0.801910
2,Toy Story 3,2010.0,Adventure|Animation|Children|Comedy|Fantasy|IMAX,3.827643,20327,0.706804
3,"Monsters, Inc.",2001.0,Adventure|Animation|Children|Comedy|Fantasy,3.837442,46036,0.696715
4,Finding Dory,2016.0,Adventure|Animation|Comedy,3.537607,5624,0.696048


## 12. Comparacion de resultados

Esta comparacion ayuda a ver si los tags aportan informacion adicional respecto al modelo basado solo en generos.

In [11]:
comparison_query = 'Matrix'

print('Recomendaciones solo con generos')
display(recommend_movies_by_genres(movies_clean, genre_feature_matrix, comparison_query, n=5, min_ratings=20))

print('Recomendaciones con generos y tags')
display(recommend_movies_by_content(comparison_query, n=5, min_ratings=20))

Recomendaciones solo con generos
Se encontraron varias coincidencias parciales. Se usara la pelicula con mas valoraciones:
                                                title   year  rating_count
                                   Matrix, The (1999) 1999.0         93808
                          Matrix Reloaded, The (2003) 2003.0         29835
                       Matrix Revolutions, The (2003) 2003.0         23699
                                Animatrix, The (2003) 2003.0          5265
                      The Matrix Resurrections (2021) 2021.0          1205
                          The Matrix Revisited (2001) 2001.0           153
                         Armitage: Dual Matrix (2002) 2002.0            27
                        A Glitch in the Matrix (2021) 2021.0            22
Return to Source: The Philosophy of The Matrix (2004) 2004.0            20
              The Matrix Revolutions Revisited (2004) 2004.0             4
Pelicula seleccionada: Matrix, The (1999)


,title,year,genres,rating_mean,rating_count,similarity_score
0,"Terminator, The",1984.0,Action|Sci-Fi|Thriller,3.902092,47131,1.0
1,Blade Runner,1982.0,Action|Sci-Fi|Thriller,4.110005,46289,1.0
2,Predator,1987.0,Action|Sci-Fi|Thriller,3.672675,20776,1.0
3,X-Men: The Last Stand,2006.0,Action|Sci-Fi|Thriller,3.208563,13991,1.0
4,Johnny Mnemonic,1995.0,Action|Sci-Fi|Thriller,2.762982,13788,1.0


Recomendaciones con generos y tags
Se encontraron varias coincidencias parciales. Se usara la pelicula con mas valoraciones:
                                                title   year  rating_count
                                   Matrix, The (1999) 1999.0         93808
                          Matrix Reloaded, The (2003) 2003.0         29835
                       Matrix Revolutions, The (2003) 2003.0         23699
                                Animatrix, The (2003) 2003.0          5265
                      The Matrix Resurrections (2021) 2021.0          1205
                          The Matrix Revisited (2001) 2001.0           153
                         Armitage: Dual Matrix (2002) 2002.0            27
                        A Glitch in the Matrix (2021) 2021.0            22
Return to Source: The Philosophy of The Matrix (2004) 2004.0            20
              The Matrix Revolutions Revisited (2004) 2004.0             4
Pelicula seleccionada: Matrix, The (1999)


,title,year,genres,rating_mean,rating_count,similarity_score
0,"Matrix Reloaded, The",2003.0,Action|Adventure|Sci-Fi|Thriller|IMAX,3.372616,29835,0.859056
1,"Matrix Revolutions, The",2003.0,Action|Adventure|Sci-Fi|Thriller|IMAX,3.236170,23699,0.812211
2,Tron,1982.0,Action|Adventure|Sci-Fi,3.334628,11973,0.552718
3,Dark City,1998.0,Adventure|Film-Noir|Sci-Fi|Thriller,3.804360,14404,0.539984
4,"Thirteenth Floor, The",1999.0,Drama|Sci-Fi|Thriller,3.365052,4850,0.528740


## 13. Conclusiones

Combinar generos y tags permite construir una representacion de contenido mas rica que la basada solo en generos. En algunos casos esto puede mejorar la precision tematica de las recomendaciones, porque los tags capturan detalles que los generos no reflejan.

Sin embargo, esta mejora tambien tiene limites. No todas las peliculas tienen tags y algunas etiquetas pueden ser ruidosas, repetitivas o poco consistentes. Por eso esta version puede ser util como extension del modelo basico, pero conviene interpretar sus resultados con cuidado.